Загружаем утилиты для быстрой подгрузки моделек

In [ ]:
import re
import time
from pathlib import Path
from utils import gemini_model_list, get_llm, get_response_text, get_concepts, google_api_key_list

Смотрим на список доступных моделей

In [2]:
gemini_model_list

['gemini-3.1-flash-lite-preview',
 'gemini-3-flash-preview',
 'gemini-2.5-pro',
 'gemini-2.5-flash',
 'gemini-2.5-flash-lite',
 'gemini-2.5-flash-lite-preview-09-2025']

### Генерация введения

Далее мы будем решать следующую задачу: имея на руках понятие (концепт), сгенерировать начало статьи. С этим прекрасно справляется данный промпт:

In [4]:
index_prompt_template = """\
Напиши введение и содержание для статьи по физике для учащихся 8 класса по заданному понятию - <CONCEPT>{concept}</CONCEPT>.

## СТРУКТУРА

### # [Заголовок]
Четкое название термина, без лишних слов.

### ## Введение

1. Увлекающее начало: нужно максимально нагнать интриги крайне коротким предложением (не больше 20 слов), которое касается повседневного опыта читателя и подводит к теме. Можно начинать со слов "почему", "зачем", "когда" и так далее, за использование клише "задумывались ли вы" и "знаете ли вы" ты получаешь штраф $2000

2. Короткое определение: 2-3 предложения максимально простым языком, без сложных терминов (они будут введены позже). Формат: "Понятие - это ..."

3. Краткий обзор понятия:
   - Почему это важная характеристика
   - В каких формах/качествах проявляется (если применимо)
   - С чем не стоит путать (но только, если есть риск смешения понятий: магнитная и электромагнитная индукция, гравитация и гравитационное ускорение)

### ## Содержание

Составь оглавление статьи. Разделы должны логически и последовательно раскрывать тему: от простого к сложному, от наблюдаемого к теоретическому. Большие разделы следует разбивать на подразделы, особенно в случае перечислений.

Формат (только оглавление, без пояснений, приведён пример оглавления):
- [1. Название раздела 1](#1-название-раздела-1)
- [2. Название раздела 2](#2-название-раздела-2)
   - [2.1. Название подраздела](#21-название-подраздела)
   - [2.2. Название подраздела](#22-название-подраздела)
- [3. Название раздела 3](#3-название-раздела-3)
... и так далее

Важно: никаких самих разделов, только введение и содержание. Текст должен быть готов к тому, что разделы будут написаны отдельно.
Очень важно: ученик 8-го класса должен узнать что-то новое или укрепить знания, но в доступной форме. Категорически ЗАПРЕЩЕНО лить бесполезную воду и бесконечные метафоры.
Ссылка на раздел оформляется так: из заголовка раздела удаляются знаки препинания, буквы приводятся к нижнему регистру, пробелы заменяются на дефисы.

<CONCEPT>{concept}</CONCEPT>
"""

### Генерация наполнения статьи

Теперь необходимо распарсить ответ предыдущей модели и выделить из неё содержание:

In [5]:
def get_index_from_text(text: str):
    import re
    # Ищем заголовок "## Содержание" напрямую
    match = re.search(r'^##\s+Содержание\s*$', text, re.MULTILINE)
    if match is None:
        raise ValueError('Блок "## Содержание" не найден в тексте')
    
    # Берём всё от "## Содержание" до следующего заголовка уровня ## или конца
    index_block = text[match.start():]
    end = re.search(r'^##\s+', index_block[3:], re.MULTILINE)  # ищем следующий ##
    if end:
        index_block = index_block[:end.start() + 3]
    
    return index_block.strip()

In [6]:
def generate_index(concept, llm):
    for attempt in range(5):
        index_prompt = index_prompt_template.format(concept=concept)
        try:
            article = get_response_text(llm.invoke(index_prompt))
        except Exception as e:
            print(e)
            return None
        try:
            get_index_from_text(article)
            return article
        except ValueError as e:
            print(f'Попытка {attempt + 1} неудачна: {e}')
    else:
        raise RuntimeError('Не удалось получить корректный ответ за 5 попыток')

и названия статей:

In [7]:
def parse_titles(index: str) -> list[str]:
    import re
    # Захватываем только пункты вида "- [1. Title]" или "- [2.1. Title]"
    pattern = r'^\s*-\s+\[(\d+(?:\.\d+)*\.\s+[^\]]+)\]'
    titles = re.findall(pattern, index, flags=re.MULTILINE)
    if not titles:
        raise ValueError('Не удалось найти пункты содержания в тексте')
    return titles

Причём если этого не получается сделать из-за того, что модель содержание сгенерировала неправильно, мы должны перегенерировать предыдущий шаг.

Далее мы должны пройтись по каждому пункту и сгенерировать часть статьи по нему. Вот промпт, который хорошо с этим справляется и пример кода

In [8]:
part_prompt_template = """\
Напиши раздел/подраздел <PART>{part}</PART> для статьи по физике для учащихся 8 класса.

Это часть большой статьи по термину <CONCEPT>{concept}</CONCEPT>. Пиши так, чтобы раздел органично вписывался в общий контекст.

### Содержание статьи:
{index}

Раздел - это заголовок, начинающийся с числа,
Подраздел - это заголовок, начинающийся с двойного числа (например 2.1, 2.2)

Твоя задача - написать раздел/подраздел - <PART>{part}</PART>

### ФОРМАТ ОТВЕТА (ОБЯЗАТЕЛЬНО):
- Ровно ОДИН заголовок уровня ### или ####
- После заголовка — ровно ОДИН абзац текста
- НИКАКИХ других заголовков (####, ### и т.д.)
- НИКАКИХ списков, подпунктов, буквенных или числовых структур

Если в ответе появится второй заголовок — ответ считается неверным.

### Требования к разделу:
- Начало файла - заголовок, начинающийся с "###", если это раздел, или с "####", если это подраздел. Далее название заголовка ОБЯЗАТЕЛЬНО с номером заголовка (Например: #### 2.3. Внесистемные единицы измерения длины)
- Если в содержании у текущего раздела есть подпункты (1.1., 1.2. и т.д.),
  ТО: 1. Игнорируй их содержание, 2. Не раскрывай их, 3. Напиши ТОЛЬКО вводный абзац. Это правило имеет ПРИОРИТЕТ над всеми остальными.
- Если это подраздел или раздел без подразделов, то раскрывай тему полностью, с разных сторон, но не забегая на темы следующих разделов.
- Если это первый раздел после введения, начинай с развития темы, заявленной во введении.
- Каждый раздел (и последний подраздел в рамках одного раздела) должен иметь логичное завершение.

## ТРЕБОВАНИЯ К РАЗДЕЛАМ
1. Последовательность: каждый следующий раздел опирается на понятия из предыдущих. Новые термины вводятся постепенно и сразу объясняются.
2. Логика раскрытия темы: от простого к сложному, от наблюдаемого к теоретическому, от макро- к микроуровню.
3. Хронология (если уместно): можно показать, как менялось понимание термина в истории науки.
4. Умеренные связки между разделами: статья должна читаться как единый текст, а не набор кусков, это обеспечивается вводными словами и словами-связками (Итак / Следовательно / Подводя итог), призывом (Рассмотрим что-то / Задумайтесь: / Тепер давайте перейдём). Не ограничивайся этим списком.
5. Примеры: если материал слишком абстрактный (например, магнетизм, электричество, квантовая физика), можно привести уместный и конкретный пример, иллюстрирующий материал.
6. Термины: если вводишь новый термин (например, "инертность"), выделяй его жирным и тут же давай пояснение.

### Стиль:
- Научно-популярный, для 8 класса
- Живой язык, метафоры, сравнения
- Без сложных формул
- Без воды и повторов
Важно: ученик должен узнать что-то новое, запрещено лить воду и сплошную поэзию
"""

In [9]:
def generate_part(concept, index, title, llm):
    part_prompt = part_prompt_template.format(
        concept=concept,
        part=title,
        index=index,
    )

    for attempt in range(3):
        try:
            part_article = get_response_text(llm.invoke(part_prompt))
        except Exception as e:
            print(e)
            return None
        
        # Считаем количество заголовков
        headers_count = sum(int(line.startswith('#')) for line in part_article.split('\n'))
        
        if headers_count == 1:
            break  # Всё ок, выходим из цикла
        
        if attempt == 2:  # Если это была последняя попытка
            raise ValueError(f'Даже с 3 попыток LLM выдает {headers_count} заголовка(ов)')

    # Проверяем формат заголовка: 
    # "n." — это раздел (одна точка после цифр)
    # "n.z." — это подраздел (две точки после цифр)
    is_title = bool(re.match(r'^\d+\.\s', title))
    is_subtitle = not is_title

    n = 3 if is_title else 4
    
    # Очищаем текст от лишних пробелов в начале/конце
    part_article = part_article.strip()
    
    # Исправляем количество решёток:
    # 1. Убираем любые существующие решётки и пробелы в начале первой строки
    clean_article = re.sub(r'^#+\s*', '', part_article)
    
    # 2. Добавляем нужное количество решёток
    part_article = f"{'#' * n} {clean_article}"

    return part_article

Пример кода для генерации статей

In [10]:
def retry_generate(func, *args):
    """Обертка для повторных попыток генерации при получении None."""
    global index_llm
    is_waited = False
    while True:
        result = func(*args, llms[index_llm])
        if result is not None:
            return result
        
        # Если None, переключаем модель
        if is_waited:
            is_waited = False
            index_llm += 1
            if index_llm >= len(llms):
                index_llm = 0
                print("Все модели исчерпаны, ожидание 120 секунд...")
                time.sleep(5 * 60)
        else:
            is_waited = True
            time.sleep(30)

# подгружаем термины для генерации
concepts = get_concepts()

# создаём папку для статей
folder_path = Path.cwd().parent / 'advanced_papers'
folder_path.mkdir(exist_ok=True)

index_llm = 0
llms = [
    get_llm('gemini-3.1-flash-lite-preview', google_api_key=google_api_key_list[0]),
    get_llm('gemini-3.1-flash-lite-preview', google_api_key=google_api_key_list[1]),
    get_llm('gemini-3.1-flash-lite-preview', google_api_key=google_api_key_list[2]),
    get_llm('gemini-3.1-flash-lite-preview', google_api_key=google_api_key_list[3]),
    get_llm('gemini-3-flash-preview', google_api_key=google_api_key_list[0]),
    get_llm('gemini-3-flash-preview', google_api_key=google_api_key_list[1]),
    get_llm('gemini-3-flash-preview', google_api_key=google_api_key_list[2]),
    get_llm('gemini-3-flash-preview', google_api_key=google_api_key_list[3]),
    get_llm('gemini-2.5-pro', google_api_key=google_api_key_list[0]),
    get_llm('gemini-2.5-pro', google_api_key=google_api_key_list[1]),
    get_llm('gemini-2.5-pro', google_api_key=google_api_key_list[2]),
    get_llm('gemini-2.5-pro', google_api_key=google_api_key_list[3]),
]

# генерируем статьи одну за одной
for concept_id, concept_name in concepts:
    path_to_paper = folder_path / f'{concept_id}.md'
    
    # проверяем, чтоб не перегенерировать статью лишний раз
    if path_to_paper.exists():
        print(f'Статья {concept_name}-{concept_id} уже сгенерирована. Пропускаем')
        continue

    # Генерируем структуру (index)
    article = retry_generate(generate_index, concept_name)
    
    with open(path_to_paper, mode='a', encoding='utf-8') as file:
        file.write(article)

        # получаем содержание и заголовки
        index = get_index_from_text(article)
        titles = parse_titles(index)

        for title in titles:
            # генерируем часть статьи
            part_article = retry_generate(generate_part, concept_name, index, title)
            file.write('\n\n' + part_article)

    print(f'Статья {concept_name}-{concept_id} успешно сгенерирована!')

Статья Вселенная-Q1 уже сгенерирована. Пропускаем
Статья пружина-Q102836 уже сгенерирована. Пропускаем
Статья термодинамическая фаза-Q104837 уже сгенерирована. Пропускаем
Статья цвет-Q1075 уже сгенерирована. Пропускаем
Статья физическая величина-Q107715 уже сгенерирована. Пропускаем
Статья инженерное дело-Q11023 уже сгенерирована. Пропускаем
Статья динамометр-Q11223329 уже сгенерирована. Пропускаем
Статья химический элемент-Q11344 уже сгенерирована. Пропускаем
Статья молекула-Q11369 уже сгенерирована. Пропускаем
Статья ускорение-Q11376 уже сгенерирована. Пропускаем
Статья энергия-Q11379 уже сгенерирована. Пропускаем
Статья закон сохранения энергии-Q11382 уже сгенерирована. Пропускаем
Статья инфракрасное излучение-Q11388 уже сгенерирована. Пропускаем
Статья сила-Q11402 уже сгенерирована. Пропускаем
Статья магнитное поле-Q11408 уже сгенерирована. Пропускаем
Статья гравитация-Q11412 уже сгенерирована. Пропускаем
Статья магнит-Q11421 уже сгенерирована. Пропускаем
Статья масса-Q11423 уже сг